# PO4AO training progress

Use the **PO4AO trainer** kernel on exao2 and run these cells. This notebook only reads the managed trainer's saved files. It never starts training, connects to INDI/shared memory, allocates a GPU, or changes the active policy. Keep your original training notebook stopped while the app trains.

If the kernel is missing, run as your normal user on exao2:
```sh
/opt/conda/envs/po4aoTrainer/bin/python -m ipykernel install --user --name po4aoTrainer --display-name "PO4AO trainer"
```
The kernel needs NumPy, SciPy and Matplotlib, already used for the training plots.

By default, follow the newest run **even if it has not produced diagnostics yet**. Set `RUN_DIRECTORY` to a particular `run-...` directory to inspect that run instead. Diagnostic plots update every configured `diagnostics_interval` (normally 50 iterations), after any engine publication finishes. The current publication result may therefore be newer than the plots. Snapshots stop updating while training is paused or stopped; always check the displayed age.


In [ ]:
# \file watch_training.ipynb
# \brief Read-only monitoring of managed PO4AO training diagnostics.
from pathlib import Path
import json
import time
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
from xapp.po4aoTrainer.observer import load_diagnostics

ARTIFACT_ROOT = Path("/data/po4aoTrainer")
RUN_DIRECTORY = None  # Or Path("/data/po4aoTrainer/run-...")
REFRESH_SECONDS = 15
AUTO_REFRESH = False  # Set True in the final cell to watch continuously.
_figure_key, _figures = None, ()


In [ ]:
def refresh():
    """Read one run, show final publication records and reuse unchanged plot figures."""
    global _figure_key, _figures
    if RUN_DIRECTORY is None:
        launches = list(ARTIFACT_ROOT.glob("run-*/launch.json"))
        run = (
            max(launches, key=lambda p: p.stat().st_mtime_ns).parent
            if launches
            else None
        )
    else:
        run = Path(RUN_DIRECTORY)
    clear_output(wait=True)
    if run is None or not run.is_dir():
        print("No trainer run found. Start learning in the GUI, then refresh.")
        return None
    print("Run:", run)

    history = {}
    metrics = run / "metrics.jsonl"
    if metrics.exists():
        with metrics.open() as source:
            for line in source:
                if line.endswith("\n") and line.strip():
                    entry = json.loads(line)
                    history[entry["training_iteration"]] = entry
    ordered = [history[key] for key in sorted(history)]
    latest = ordered[-1] if ordered else {}
    if latest:
        print("Latest diagnostic iteration:", latest["training_iteration"])
        print(
            "Publication:",
            latest.get("publication_status"),
            latest.get("published_model_id"),
        )
        print("Reasons:", "; ".join(latest.get("publication_reasons", [])) or "none")
        print(
            "Replay model:",
            latest.get("active_model_id"),
            "| runtime model at sampling:",
            latest.get("runtime_active_model_id"),
        )
        for key in (
            "observation_rms",
            "measured_cost_per_step",
            "candidate_cost_ratio",
            "training_seconds",
            "replay_seconds",
            "diagnostics_seconds",
            "export_seconds",
            "activation_seconds",
        ):
            print(f"{key}: {latest.get(key, 'not recorded by this version')}")
        if len(ordered) > 1:
            previous = ordered[-2]
            seconds = (
                latest["timestamp_unix_ns"] - previous["timestamp_unix_ns"]
            ) / 1e9
            iterations = latest["training_iteration"] - previous["training_iteration"]
            print(
                f"Wall seconds/iteration between diagnostics: {seconds / iterations:.3f} (includes waits/pauses)"
            )
        age = max(0, time.time() - latest["timestamp_unix_ns"] / 1e9)
        print(f"Diagnostic measurement age: {age:.1f} s")
    else:
        print("Waiting for the first diagnostic iteration (normally iteration 50).")

    snapshots = sorted((run / "diagnostics").glob("diagnostics-*.npz"))
    if not snapshots:
        print(
            "No completed plot snapshot in this run yet; a publication may still be building."
        )
        return latest
    path = snapshots[-1]
    try:
        key = (path, path.stat().st_mtime_ns)
        if key != _figure_key:
            snapshot = load_diagnostics(path)
            for figure in _figures:
                plt.close(figure)
            _figures = snapshot.plot()
            _figure_key = key
        print("Plots:", path.name)
        print(f"Plot file age: {max(0, time.time() - key[1] / 1e9):.1f} s")
        for figure in _figures:
            display(figure)
            plt.close(figure)
    except FileNotFoundError:
        print("Snapshot rotated during this refresh; refresh again.")
    return latest


latest_record = refresh()


## Reading the plots

- **Residual RMS, measured observation cost and residual PSD:** compare under matched flux, modulation, reconstructor, turbulence and gains. Lower residuals with acceptable commands are the practical objective. Compare settled windows, not the warmup/noise or switching transients.
- **Predicted candidate cost / fallback:** below one means the learned dynamics predict an improvement. Confirm it using measured residuals; this is not measured closed-loop gain.
- **Dynamics validation and predicted versus measured cost:** persistent disagreement can explain why apparently successful optimization fails to improve the instrument.
- **Publication and model IDs:** distinguish a training candidate from the engine actually acknowledged by the runtime. An engine ID alone does not verify downstream CACAO routing.
- **Timing:** `training_seconds` covers conversion/optimization within one sampled iteration; `replay_seconds` covers shared-memory snapshot acquisition. The other fields separate diagnostics, export and activation waiting. File-save time is included only in the wall-clock interval, which also includes operator pauses. These are sampled timings, not averages of every gradient update.

To compare controllers, pause learning and retain the policy, then use live instrument measurements for matched PO4AO/integrator comparisons. This notebook's saved plots do not refresh while paused. Do not Reset just to perform that comparison.

## Optional automatic refresh

Set `AUTO_REFRESH = True` below and execute the cell. Use **Interrupt Kernel** to stop watching; this does not stop the trainer. Plot computations are reused until the snapshot changes. Leave it False for manual refresh with `latest_record = refresh()`.


In [ ]:
AUTO_REFRESH = False
try:
    while AUTO_REFRESH:
        latest_record = refresh()
        time.sleep(max(1, REFRESH_SECONDS))
except KeyboardInterrupt:
    print("Stopped watching. The trainer continues independently.")
